In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import math

# Bài 1

Import ảnh và resize tất cả các ảnh thành cùng 1 kích thước 256x256.

In [ ]:
imgs = []
for i in range (0, 8):
  path = f"img\\anh{i}.png"
  img = np.array(Image.open(path).convert("L").resize((256, 256)), dtype=float)
  imgs.append(img)

### 1. Biểu diễn dữ liệu thành ma trận:

In [ ]:
# Moi anh duoc lam phang thanh vecto 1 chieu
X = np.array([img.flatten() for img in imgs])
print(X.shape)

- Mỗi hàng của ma trận đại diện cho 1 bức ảnh (8 hàng là 8 ảnh).
- Mỗi cột của ma trận đại diện cho giá trị pixel tại 1 vị trí cố định trên tất cả các ảnh (65536 cột nghĩa là mỗi ảnh chứa 65536 pixel).

### 2. Phép toán cơ bản:

In [ ]:
# Tinh trung binh theo cot
X_mean = np.mean(X, axis=0)

# Tru tung hang cua X cho X_mean
Y = X - X_mean
print(X.shape, X_mean.shape, Y.shape)

Vecto X_mean được tự động sao chép thành 8 hàng (broadcasting) để phù hợp kích thước với ma trận X khi thực hiện phép trừ.

### 3. Cosine similarity:

In [ ]:
def cosine_similarity(X, Y=None):
  if Y is None:
    Y = X
  Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
  Yn = Y / np.linalg.norm(Y, axis=1, keepdims=True)
  return Xn @ Yn.T

Hàm trả về 1 ma trận S, với phần tử Sᵢⱼ là giá trị cosine similarity giữa vecto ảnh X[i] và Y[j].

### 4. Hàm truy vấn:

In [ ]:
def search(query, top_k=3):
  query = query.reshape(1, 65536)
  S = cosine_similarity(X, query).flatten()
  idx_lst = np.argpartition(S, -top_k)[-top_k:]
  idx_score_lst = [(idx, S[idx]) for idx in idx_lst]
  idx_score_lst.sort(key=lambda p: p[1], reverse=True)
  return idx_score_lst

# Chay thu ham search voi query la anh7.png
query = X[4, :]
res = search(query)
for rank, (idx, score) in enumerate(res):
  print(f"Rank {rank}: Index {idx}, score {score}")

### 5. Nhận xét:

In [ ]:
S = cosine_similarity(X, X)
min_idx = np.argmin(S)
print(f"Min cosine similarity pair: {min_idx // 8} and {min_idx % 8}")
S[np.diag_indices_from(S)] = 0
max_idx = np.argmax(S)
print(f"Max cosine similarity pair: {max_idx // 8} and {max_idx % 8}")

Thực hiện phép toán này trên ma trận X với chính nó để tìm cosine similarity giữa tất cả các cặp vecto ảnh. Nhận xét:
- Cặp khác biệt nhất là ảnh 1 và 6.
- Cặp giống nhất là ảnh 3 và 7.

Kết quả khớp với trực giác.

# Bài 2

### 1. Biến đổi tuyến tính:

In [ ]:
def draw(M):
  v1 = M[:, 0]
  v2 = M[:, 1]
  fig, ax = plt.subplots(figsize=(6, 6))
  ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
  ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
  ax.quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1, color='crimson')
  ax.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1, color='royalblue')
  ax.set_xlim([-1.5, 2])
  ax.set_ylim([-0.5, 2])
  ax.set_aspect('equal')
  ax.grid(True, linestyle=':', alpha=0.6)
  plt.show()

Vẽ ma trận gốc

In [ ]:
M = np.array([
    [1, 0],
    [0, 1]
])
draw(M)

Xoay ma trận 45 độ

In [ ]:
theta = np.radians(45)
R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])
M_rotate = R @ M
draw(M_rotate)

Scale ma trận theo tỉ lệ: thu nhỏ trục X 0.5 lần, kéo giãn trục Y 1.5 lần

In [ ]:
S = np.array([
    [0.5, 0.0],
    [0.0, 1.5]
])
M_scale = S @ M
draw(M_scale)

### 2. Nén ảnh bằng SVD:

In [ ]:
M = np.array(Image.open("img\\dog.jpg").convert("L"), dtype=float)
U, S, Vt = np.linalg.svd(M, full_matrices=False)

def reconstruct(k):
  return (U[:, :k] * S[:k]) @ Vt[:k, :]

plt.figure(figsize=(15, 5))
val = [5, 20, 50]
M_ks = []
for idx, k in enumerate(val):
  M_k = reconstruct(k)
  plt.subplot(1, 3, idx + 1)
  plt.imshow(M_k, cmap='gray')
  plt.title(f"k = {k}", fontsize=18)
  plt.axis('off')
  M_ks.append((k, M_k))
plt.tight_layout()
plt.show()

### 3. Đánh giá:

Sai số bình phương của phép xấp xỉ M bằng M_k được tính bằng tổng bình phương các giá trị kì dị bị bỏ đi (từ k + 1 đến giá trị cuối cùng).

In [ ]:
H, W = M.shape
original_size = H * W
for k, M_k in M_ks:
  print(f"Voi k = {k}:")
  # Tinh sai so
  error = math.sqrt(np.sum(S[k:] ** 2))
  print(f"- Sai so tai tao: {error}")
  # Tinh ti le nen
  compressed_size = k * (H + W + 1)
  ratio = compressed_size / original_size
  print(f"- Ti le nen: {ratio}")

In [ ]:
# Ve do thi sai so theo k (logscale)
error = np.zeros_like(S)
for i in range(S.shape[0]):
  error[i] = math.sqrt(np.sum(S[i:] ** 2))
plt.semilogy(error)
plt.xlabel("k")
plt.ylabel("Sai số (logscale)")
plt.show()

### 4. Nhận xét:

- Với k = 20 thì ảnh vẫn chấp nhận được về mặt trực quan.
- SVD giúp giảm chiều dữ liệu, chỉ giữ lại các đặc trưng quan trọng nhất (tương ứng với các giá trị kì dị lớn nhất). Nhờ vậy có thể tiết kiệm bộ nhớ lưu trữ mà không làm mất đi quá nhiều thông tin ảnh, giúp các mô hình AI chạy nhanh hơn.

### Đồ thị năng lượng tích lũy của các giá trị kì dị

In [ ]:
total = np.sum(S ** 2)
reserved_energy = np.zeros_like(S)
for i in range(S.shape[0]):
  reserved_energy[i] = np.sum(S[:i+1] ** 2) / total
plt.plot(reserved_energy)
plt.xlabel("k")
plt.ylabel("Tỉ lệ năng lượng tích lũy")
plt.show()

In [ ]:
k_90 = np.where(reserved_energy >= 0.90)[0][0] + 1
print(f"Gia tri k giu >= 90% thong tin: k = {k_90}")